# Dataset Exploration

This notebook follows `L.ipynb` exactly for dataset structure: `mini_dataset/<class_name>/<image>`. It verifies folders, counts images per class, filters classes using `MIN_IMAGES`, sorts class names, and creates the numeric `label_map`.

In [ ]:
from pathlib import Path
import ast
import json
import os
import random
from collections import Counter

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
import seaborn as sns

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

# Step 3 from L.ipynb: mount Google Drive when running inside Colab.
if Path('/content').exists():
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as exc:
        print('Google Drive mount skipped:', exc)

# L.ipynb reference path: /content/drive/MyDrive/lipy/mini_dataset
# Local fallback path: <repo>/data/mini_dataset, then <repo>/data.
COLAB_DATASET_DIR = Path('/content/drive/MyDrive/lipy/mini_dataset')
LOCAL_MINI_DATASET_DIR = PROJECT_ROOT / 'data' / 'mini_dataset'
LOCAL_DATA_DIR = PROJECT_ROOT / 'data'

if COLAB_DATASET_DIR.exists():
    DATASET_DIR = COLAB_DATASET_DIR
elif LOCAL_MINI_DATASET_DIR.exists():
    DATASET_DIR = LOCAL_MINI_DATASET_DIR
else:
    DATASET_DIR = LOCAL_DATA_DIR

MIN_IMAGES = 25
IMG_SIZE = 64
BATCH_SIZE = 32
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'training'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_ARTIFACT_DIR = PROJECT_ROOT / 'outputs' / 'models'
METRICS_DIR = PROJECT_ROOT / 'outputs' / 'metrics'
MODEL_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

DRIVE_MODEL_ARTIFACT_DIR = Path('/content/drive/MyDrive/lipy_models')
if Path('/content/drive/MyDrive').exists():
    DRIVE_MODEL_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
else:
    DRIVE_MODEL_ARTIFACT_DIR = None

RUN_ID = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')
DEPLOY_MODEL_PATH = PROJECT_ROOT / 'backend' / 'models' / 'odia_ocr_cnn.keras'

def model_artifact_path(model_family):
    safe_family = str(model_family).strip().lower().replace(' ', '_')
    return MODEL_ARTIFACT_DIR / f'lipy_odia_ocr_{safe_family}_{RUN_ID}.keras'


def save_model_to_local_and_drive(model, model_family):
    local_path = model_artifact_path(model_family)
    model.save(local_path)
    print('Saved local model:', local_path)

    drive_path = None
    if DRIVE_MODEL_ARTIFACT_DIR is not None:
        drive_path = DRIVE_MODEL_ARTIFACT_DIR / local_path.name
        model.save(drive_path)
        print('Saved Google Drive model:', drive_path)
    else:
        print('Google Drive model save skipped: Drive is not mounted/available.')

    return local_path, drive_path


def latest_local_model():
    models = sorted(MODEL_ARTIFACT_DIR.glob('lipy_odia_ocr_*.keras'), key=lambda p: p.stat().st_mtime, reverse=True)
    if models:
        return models[0]
    if DEPLOY_MODEL_PATH.exists():
        return DEPLOY_MODEL_PATH
    raise FileNotFoundError('No model found in outputs/models or backend/models.')

print('Project Root:', PROJECT_ROOT)
print('Dataset Path:', DATASET_DIR)
print('TensorFlow:', tf.__version__)
print('GPU Devices:', tf.config.list_physical_devices('GPU'))

In [ ]:
def list_class_folders(dataset_dir):
    if not dataset_dir.exists():
        raise FileNotFoundError(f'Dataset folder does not exist: {dataset_dir}')
    return [p.name for p in dataset_dir.iterdir() if p.is_dir()]


def count_images(class_folder):
    folder_path = DATASET_DIR / class_folder
    return len([p for p in folder_path.iterdir() if p.is_file()])

# Step 5 from L.ipynb: list all folders/classes.
classes = list_class_folders(DATASET_DIR)
print('Total Classes:', len(classes))
print(classes[:10])

# Step 7 from L.ipynb: keep classes with enough images.
valid_classes = []
for folder in classes:
    image_count = count_images(folder)
    if image_count >= MIN_IMAGES:
        valid_classes.append(folder)
        print(f'{folder} -> {image_count} images')

# Step 8 from L.ipynb: sorting gives consistent label ordering.
valid_classes.sort()
print('Valid Classes:', len(valid_classes))
print(valid_classes)

# Step 9 from L.ipynb: class-name to integer label.
label_map = {class_name: idx for idx, class_name in enumerate(valid_classes)}
reverse_label_map = {idx: class_name for class_name, idx in label_map.items()}
num_classes = len(valid_classes)
print('Number of Classes:', num_classes)
print(label_map)

if num_classes < 2:
    raise ValueError('Need at least two valid class folders. Check DATASET_DIR and MIN_IMAGES.')

(OUTPUT_DIR / 'label_map.json').write_text(json.dumps(label_map, indent=2, ensure_ascii=False), encoding='utf-8')

In [ ]:
counts = pd.Series({class_name: count_images(class_name) for class_name in classes}).sort_index()
valid_counts = counts.loc[valid_classes]

summary = pd.DataFrame({
    'class_name': counts.index,
    'image_count': counts.values,
    'is_valid': counts.index.isin(valid_classes),
})
display(summary)
summary.to_csv(OUTPUT_DIR / 'dataset_summary.csv', index=False)

plt.figure(figsize=(16, 5))
valid_counts.plot(kind='bar')
plt.title('Images per valid class')
plt.ylabel('Images')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 10))
shown = 0
for class_name in valid_classes:
    class_path = DATASET_DIR / class_name
    image_files = [p for p in class_path.iterdir() if p.is_file()]
    if not image_files:
        continue
    img = cv2.imread(str(image_files[0]), cv2.IMREAD_GRAYSCALE)
    if img is None:
        continue
    shown += 1
    plt.subplot(6, 6, shown)
    plt.imshow(img, cmap='gray')
    plt.title(class_name, fontsize=8)
    plt.axis('off')
    if shown == 36:
        break
plt.tight_layout()
plt.show()